### Import libraries

In [1]:
# set current directory to the main project directory
import os

curr_dir = os.getcwd()
print("Current directory: ", curr_dir)
root_dir = os.path.dirname(curr_dir)  # goes up to ashish-ai-lab/
os.chdir(root_dir)
print("Changed to Root directory: ", os.getcwd())

Current directory:  e:\Projects\ashish-ai-lab\prompting-techniques
Changed to Root directory:  e:\Projects\ashish-ai-lab


In [2]:
import os
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from collections import Counter
from dotenv import load_dotenv

load_dotenv()

# importing functions
from src.utils.llm_funcs import llm_setup
from src.prompts.system_prompts import (
    zero_shot_customer_support_ticket_classifier_system_prompt,
    few_shot_customer_support_ticket_classifier_system_prompt,
    cot_shot_customer_support_ticket_classifier_system_prompt,
    react_customer_support_template,
)
from src.utils.helper_funcs import parse_category, get_order_status, get_account_balance

#### LLM Setup

In [3]:
llm = llm_setup(api_key=os.getenv("OPENAI_API_KEY"))

### Prompt Techniques
#### 1. Zero-shot prompting
Providing the model with a task without any examples.

In [4]:
# creating a prompt template for zero-shot classification of customer support tickets
task_prompt = zero_shot_customer_support_ticket_classifier_system_prompt

zero_shot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", task_prompt),
])

In [ ]:
# assuming the ticket text is provided as input to the llm
ticket_text = """I tried to log in this morning but keep getting an error saying my password is incorrect. I already tried resetting it twice."""

# adding the ticket text to the prompt template
zero_shot_prompt_template.messages.append(HumanMessagePromptTemplate.from_template("Ticket: {ticket_text}"))

# invoking the prompt template with the ticket text
formatted = zero_shot_prompt_template.invoke({"ticket_text": ticket_text})

# printing the formatted messages
for msg in formatted.messages:
    print(f"{msg.type}: {msg.content}")
    print("-" * 50)

# invoking the llm with the formatted messages
response = llm.invoke(formatted)
print("\n--- Response from LLM ---")
print(response.content)

system: 
[ROLE]
You are a customer support ticket classifier for an ecommerce company. 

[CONTEXT]
You will receive support tickets submitted by customers. 
Your job is to classify these support tickets into exactly one category so that the right support team can handle it further.

[CATEGORIES]
- Billing: questions about invoices, payments, refunds
- Technical: bugs, errors, login issues, performance problems
- Account: password reset, profile updates, account deletion
- Shipping: delivery status, lost packages, address changes
- General: anything that does not fit the above

[INSTRUCTIONS]
- Read the ticket carefully.
- Identify the core issue customer mentioned in the ticket.
- Assign exactly one category from the list of categories provided above.
- If the ticket does not fit into any of the categories, respond with "Uncategorized".

[OUTPUT FORMAT]
Output only the category name without any additional text, explanation or commentary.
Example of correct output: Technical
Example of 

#### 2. Few shot prompting
Providing the model with a task along with a few examples to learn from.

In [6]:
# creating a prompt template for few-shot classification of customer support tickets
task_prompt = few_shot_customer_support_ticket_classifier_system_prompt

few_shot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", task_prompt),
])

In [ ]:
# assuming the ticket text is provided as input to the llm
ticket_text = """I tried to log in this morning but keep getting an error saying my password is incorrect. I already tried resetting it twice."""

# adding the ticket text to the prompt template
few_shot_prompt_template.messages.append(HumanMessagePromptTemplate.from_template("Ticket: {ticket_text}"))

# invoking the prompt template with the ticket text
formatted = few_shot_prompt_template.invoke({"ticket_text": ticket_text})

# printing the formatted messages
for msg in formatted.messages:
    print(f"{msg.type}: {msg.content}")
    print("-" * 50)

# invoking the llm with the formatted messages
response = llm.invoke(formatted)
print("\n--- Response from LLM ---")
print(response.content)

system: 
[ROLE]
You are a customer support ticket classifier for an ecommerce company. 

[CONTEXT]
You will receive support tickets submitted by customers. 
Your job is to classify these support tickets into exactly one category so that the right support team can handle it further.

[CATEGORIES]
- Billing: questions about invoices, payments, refunds
- Technical: bugs, errors, login issues, performance problems
- Account: password reset, profile updates, account deletion
- Shipping: delivery status, lost packages, address changes
- General: anything that does not fit the above

[INSTRUCTIONS]
- Read the ticket carefully.
- Identify the core issue customer mentioned in the ticket.
- Assign exactly one category from the list of categories provided above.
- If the ticket does not fit into any of the categories, respond with "Uncategorized".

[EXAMPLES]
Ticket: I was charged twice for my subscription this month.
Category: Billing

Ticket: I need to update the email address on my account.
Ca

#### 3. Chain of thought prompting
Providing the model with a task and asking it to reason step by step before giving the final answer.

In [8]:
# creating a prompt template for chain of thought (CoT)-shot classification of customer support tickets
task_prompt = cot_shot_customer_support_ticket_classifier_system_prompt

cot_shot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", task_prompt),
])

In [ ]:
# assuming the ticket text is provided as input to the llm
ticket_text = """I tried to log in this morning but keep getting an error saying my password is incorrect. I already tried resetting it twice."""

# adding the ticket text to the prompt template
cot_shot_prompt_template.messages.append(HumanMessagePromptTemplate.from_template("Ticket:{ticket_text}"))

# invoking the prompt template with the ticket text
formatted = cot_shot_prompt_template.invoke({"ticket_text": ticket_text})

# printing the formatted messages
for msg in formatted.messages:
    print(f"{msg.type}: {msg.content}")
    print("-" * 50)

# invoking the llm with the formatted messages
response = llm.invoke(formatted)
print("\n--- Response from LLM ---")
print(response.content)

system: 
[ROLE]
You are a senior customer support ticket classifier for an ecommerce company.

[CONTEXT]
You will receive support tickets submitted by customers. Tickets may be ambiguous, span multiple issues, use emotional language, or omit key details.
Your job is to determine the single most actionable category so the right support team can resolve the customer's primary problem.

[CATEGORIES]
- Billing: questions about invoices, payments, refunds, unexpected charges, pricing disputes
- Technical: bugs, errors, login issues, app crashes, performance problems, integration failures
- Account: password reset, profile updates, account deletion, verification, access permissions
- Shipping: delivery status, lost or damaged packages, wrong items received, address changes, carrier disputes
- General: feedback, compliments, product questions, or anything that does not fit the above

[REASONING FRAMEWORK]
Before assigning a category, think and reason through the ticket in steps like (step-1, 

* With CoT, the LLM changed the category from Account to Technical. This is what chain of thought can do.

#### 4. Self-consistency
Generating multiple diverse reasoning paths and selecting the most consistent answer
* From zero shot and few shot, we got the category "Account" but with CoT, we got "Technical". Different responses for same query. And this is exactly why we need self-consistency in such situations. 
* We will run the same CoT prompt on same query multiple times, and pick the category which appears the most numbers of times because that is likely to be the right pick.

In [10]:
# for self-consistency technique, we can set a temperature value to generate diverse reasoning paths
llm = llm_setup(api_key=os.getenv("OPENAI_API_KEY"), temperature = 0.6)

We will use same CoT prompt template inside self-consistency with higher temperature. This way each run reasons differently, which is exactly what we want

In [11]:
# creating a prompt template for chain of thought (CoT)-shot classification of customer support tickets
task_prompt = cot_shot_customer_support_ticket_classifier_system_prompt

cot_shot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", task_prompt),
])

In [ ]:
# assuming the ticket text is provided as input to the llm
# changed the ticket text to a more complex one for testing self-consistency
ticket_text = """I placed an order last week, the tracking says delivered but I never got it, and on top of that I was charged twice for the same order. /
I've tried logging in to check my order history but my account keeps getting locked out."""

# adding the ticket text to the prompt template
cot_shot_prompt_template.messages.append(HumanMessagePromptTemplate.from_template("Ticket: {ticket_text}"))

# invoking the prompt template with the ticket text
formatted = cot_shot_prompt_template.invoke({"ticket_text": ticket_text})

print(f"Human: {ticket_text}")

# n_samples is the number of diverse reasoning paths we want to generate for self-consistency
n_samples = 5
categories = []
reasoning_logs = []

for i in range(n_samples):
    response = llm.invoke(formatted)
    category = parse_category(response.content)

    if category:
        categories.append(category)
        reasoning_logs.append({
            "run": i+1,
            "reasoning": response.content.split("Category:")[0].strip(),
            "category": category
        })

# Majority vote logic to determine the final category based on the most frequent category from the diverse reasoning paths
vote_counts = Counter(categories)
final_category = vote_counts.most_common(1)[0][0]
confidence = vote_counts[final_category] / n_samples # 

print(f"\n--- Final Category ---")
print(f"Category: {final_category}")
print(f"Confidence: {confidence:.1%}")

Human: I placed an order last week, the tracking says delivered but I never got it, and on top of that I was charged twice for the same order. /
I've tried logging in to check my order history but my account keeps getting locked out.

--- Final Category ---
Category: Billing
Confidence: 60.0%


In [13]:
reasoning_logs

[{'run': 1,
  'reasoning': "Reasoning: \nStep-1: Identifying issues mentioned in the ticket\n- The customer's order was not delivered despite tracking showing delivery.\n- The customer was charged twice for the same order.\n- The customer's account keeps getting locked out when trying to access order history.\n\nStep-2: Determining the root cause of the ticket\n- The root cause appears to be an issue with the order fulfillment and billing.\n\nStep-3: Evaluating which category best fits the ticket\n- The issues of missing delivery and double charge suggest a billing issue, but the locked out account could also indicate a technical issue.\n- However, the billing issue is more prominent and action-oriented as it directly impacts the customer's financial situation.\n\nStep-4: Resolving any ambiguity if multiple categories seem applicable\n- While the locked out account is a secondary issue, it's likely a symptom of the billing issue rather than a standalone problem.\n\nStep-5: State your c

#### 5. ReAct prompting
ReAct prompting makes an LLM alternate between reasoning (thinking through the problem step-by-step) and acting (calling a tool or taking an action), using each action's result to inform the next thought — repeating until it reaches a final answer.

In [3]:
llm = llm_setup(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
# Tool registry — maps tool names to functions
TOOLS = {
    "get_order_status": get_order_status,
    "get_account_balance": get_account_balance,
}

TOOL_DESCRIPTIONS = """
- get_order_status(order_id): Returns the current status of an order.
  Use when the customer asks about an order, delivery, or shipment.
  Input: order ID (format: ORD-XXX)

- get_account_balance(account_id): Returns account balance and payment history.
  Use when the customer asks about charges, payments, or their balance.
  Input: account ID (format: ACC-XXX)
"""

react_prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(react_customer_support_template),
    HumanMessagePromptTemplate.from_template("Customer ID: {customer_id} \nCustomer Query: {query}"),
])

formatted = react_prompt_template.invoke({
    "tool_descriptions": TOOL_DESCRIPTIONS,
    "customer_id": "ACC-101",
    "query": "Can you tell me the status of my order ORD-002 and whether I have any pending charges on my account?"
})

# printing the formatted messages
for msg in formatted.messages:
    print(f"{msg.type}: {msg.content}")
    print("-" * 50)

# invoking the llm with the formatted messages
response = llm.invoke(formatted)
print("\n--- Response from LLM ---")
print(response.content)
    

system: 
[ROLE]
You are a customer support agent for an e-commerce company. You help customers by looking up their order and account information.

[TOOLS]
You have access to the following tools:

- get_order_status(order_id): Returns the current status of an order.
  Use when the customer asks about an order, delivery, or shipment.
  Input: order ID (format: ORD-XXX)

- get_account_balance(account_id): Returns account balance and payment history.
  Use when the customer asks about charges, payments, or their balance.
  Input: account ID (format: ACC-XXX)


[INSTRUCTIONS]
Use the following loop to answer the customer's question:

Thought: reason about what you know and what you need to find out
Action: tool_name(input)
Observation: <result will be provided here>
... repeat Thought/Action/Observation as needed ...
Thought: I now have enough information to answer
Final Answer: <your response to the customer>

Rules:
- Always start with a Thought
- Only call one tool per Action step
- Wait